# Tutorial 07 — TikZ Backend

**TikZ** is the de-facto standard for drawing in LaTeX documents.  
maxplotlib can render your figures as native TikZ code via the `tikzfigure` backend,
which wraps the [`tikzfigure`](https://github.com/max-models/tikzfigure) Python package.

This tutorial covers **two complementary workflows**:

| Workflow | When to use |
|---|---|
| **Canvas → TikZ** | Quick way to turn data plots into LaTeX-ready TikZ code |
| **`tikzfigure` API directly** | Full control — nodes, shapes, annotations, arcs, colours … |

**Prerequisites**

```bash
pip install tikzfigure
```

To actually *render* the figure (not just generate code) you also need `pdflatex` installed on your system.

In [ ]:
import numpy as np
import tikzfigure as tz
from maxplotlib import Canvas

---
## Part 1 — Canvas → TikZ

The fastest path: build a plot with the standard Canvas API, then pass `backend='tikzfigure'` to get a `TikzFigure` object back.

### 1.1 Basic usage

In [ ]:
x = np.linspace(0, 2 * np.pi, 60)

canvas, ax = Canvas.subplots(width="10cm", ratio=0.6)
ax.plot(x, np.sin(x), label="sin", color="steelblue", line_width=1.5)
ax.plot(x, np.cos(x), label="cos", color="tomato", line_width=1.2)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Trigonometric functions")

# backend='tikzfigure' returns a TikzFigure object
tikz = canvas.plot(backend="tikzfigure")
print(type(tikz))

### Plotly preview

Before exporting to TikZ, you can preview the same `Canvas` interactively in a notebook using the Plotly backend:



In [ ]:
canvas.show(backend="plotly")

### 1.2 Inspecting the generated LaTeX

`tikz.generate_tikz()` returns the raw LaTeX source string.  
Each data line becomes a `\draw` command connecting coordinate pairs.

In [ ]:
code = tikz.generate_tikz()
print(code)

### 1.3 TikZ-specific kwargs

The TikZ backend passes extra keyword arguments straight to `tikzfigure.draw()`.  
Use **`line_width=`** (not matplotlib's `linewidth=`) to control stroke thickness.

In [ ]:
canvas2, ax2 = Canvas.subplots(width="10cm", ratio=0.5)
ax2.plot(x, np.sin(x), color="navy", line_width=0.5, label="thin")
ax2.plot(x, np.sin(x) + 0.5, color="steelblue", line_width=1.5, label="medium")
ax2.plot(x, np.sin(x) + 1.0, color="royalblue", line_width=3.0, label="thick")
ax2.set_xlabel("x")
ax2.set_title("Line width comparison")

tikz2 = canvas2.plot(backend="tikzfigure")
print(tikz2.generate_tikz())

### 1.4 Layer-aware TikZ output

Assign data to layers with `layer=N`.  
The TikZ backend respects the layer filter — useful for generating incremental reveal figures (e.g. in Beamer).

In [ ]:
canvas3, ax3 = Canvas.subplots(width="10cm", ratio=0.55)
ax3.plot(x, np.sin(x), color="steelblue", line_width=1.5, layer=0, label="sin")
ax3.plot(x, np.cos(x), color="tomato", line_width=1.5, layer=1, label="cos")
ax3.plot(
    x, np.sin(x) * np.cos(x), color="seagreen", line_width=1.0, layer=2, label="sin·cos"
)

# All layers available on the canvas
print("Available layers:", canvas3.layers)

# Render only layer 0 — one \draw command
tikz_l0 = canvas3.plot(backend="tikzfigure", layers=[0])
print("\n--- Layer 0 only ---")
print(f"\\draw count: {tikz_l0.generate_tikz().count(chr(92) + 'draw')}")

# Render layers 0 and 1
tikz_l01 = canvas3.plot(backend="tikzfigure", layers=[0, 1])
print("\n--- Layers 0 & 1 ---")
print(f"\\draw count: {tikz_l01.generate_tikz().count(chr(92) + 'draw')}")

### 1.5 Saving TikZ code to a file

You can embed the generated code directly in a LaTeX document:

In [ ]:
tikz_all = canvas3.plot(backend="tikzfigure")

with open("figure.tex", "w") as f:
    f.write(tikz_all.generate_tikz())

print("Saved figure.tex")

# In your LaTeX document:
# \input{figure.tex}
# or wrap it:
# \begin{figure}[h]
#   \centering
#   \input{figure.tex}
#   \caption{My caption}
# \end{figure}

### 1.6 Rendering the figure (requires `pdflatex`)

If `pdflatex` is installed, `tikz.show()` compiles the code and opens the PDF:

In [ ]:
# Requires pdflatex:
tikz_all.show(transparent=False)

### 1.7 Canvas → TikZ limitations

| Feature | Supported? |
|---|---|
| Line plots (`ax.plot`) | ✅ |
| Layer filtering | ✅ |
| `line_width=` kwarg | ✅ |
| Multiple subplots | ❌ (raises `NotImplementedError`) |
| `ax.scatter`, `ax.bar` | ❌ (silently ignored) |
| `ax.fill_between` | ❌ |
| Axis labels / titles | ❌ (TikZ has no axis frame by default) |

For anything beyond line plots, use the `tikzfigure` API directly (Part 2 below).

---
## Part 2 — The `tikzfigure` API

The `tikzfigure` package gives you a full Python interface to TikZ primitives.
You build figures by adding nodes, paths, shapes, and annotations, then
call `generate_tikz()` (or `show()`) to obtain the output.

```python
import tikzfigure as tz
tf = tz.TikzFigure()
```

### 2.1 Drawing paths with `draw()`

`tf.draw(nodes, ...)` produces a `\draw` path through a list of `(x, y)` coordinates.

In [ ]:
tf = tz.TikzFigure()

x = np.linspace(0, 2 * np.pi, 60)
sin_nodes = [(float(xi), float(np.sin(xi))) for xi in x]
cos_nodes = [(float(xi), float(np.cos(xi))) for xi in x]

tf.draw(sin_nodes, color="steelblue", line_width=1.5)
tf.draw(cos_nodes, color="tomato", line_width=1.2)

print(tf.generate_tikz())

### 2.2 Straight line segments with `line()`

`tf.line(start, end, ...)` is a convenience wrapper for a two-point path.  
The `arrows` parameter adds arrowheads.

In [ ]:
tf2 = tz.TikzFigure()

# Baseline
tf2.line((0, 0), (2 * np.pi, 0), color="gray", dash_pattern="on 3pt off 3pt")

# Arrow showing direction
tf2.line((0, -1.2), (0, 1.2), color="black", arrows="->", line_width=0.8)
tf2.line((-0.2, 0), (2 * np.pi + 0.2, 0), color="black", arrows="->", line_width=0.8)

# The curve
tf2.draw(sin_nodes, color="steelblue", line_width=1.5)

print(tf2.generate_tikz())

### 2.3 Rectangles, circles, and arcs

In [ ]:
tf3 = tz.TikzFigure()

# Bounding rectangle (coordinate space for context)
tf3.rectangle((0, -1.2), (2 * np.pi, 1.2), draw="gray!40", fill="gray!5")

# Circle at the origin
tf3.circle((0, 0), radius=0.15, fill="red!60", draw="red")

# Circle at peak of sine
tf3.circle((np.pi / 2, 1.0), radius=0.12, fill="steelblue", draw="none")

# Arc (quarter circle)
tf3.arc(
    (0.4, 0),
    start_angle=0,
    end_angle=90,
    radius=0.4,
    draw="green!60!black",
    line_width=1.0,
)

# The curve on top
tf3.draw(sin_nodes, color="steelblue", line_width=1.5)

print(tf3.generate_tikz())

### 2.4 Nodes — text labels and markers

`add_node()` places a text label (optionally inside a shape) at an `(x, y)` position.

In [ ]:
tf4 = tz.TikzFigure()
tf4.draw(sin_nodes, color="steelblue", line_width=1.5)

# Plain text label
tf4.add_node(np.pi / 2, 1.15, content=r"$\max$", color="steelblue")

# Boxed label
tf4.add_node(
    3 * np.pi / 2,
    -1.15,
    content=r"$\min$",
    shape="rectangle",
    fill="tomato!20",
    draw="tomato",
    inner_sep="2pt",
)

# Circle marker at zero-crossing
tf4.add_node(
    np.pi, 0, shape="circle", fill="white", draw="steelblue", minimum_size="0.18cm"
)

print(tf4.generate_tikz())

### 2.5 Custom colours with `colorlet()`

TikZ colour mixing syntax (`blue!70!white`) lets you define reusable named colours.

In [ ]:
tf5 = tz.TikzFigure()

# Define named colours
tf5.colorlet("myblue", "blue!70!white")
tf5.colorlet("myred", "red!80!black")
tf5.colorlet("myfill", "blue!10!white")

# Use them in draw calls
tf5.draw(sin_nodes, color="myblue", line_width=1.5)
tf5.draw(cos_nodes, color="myred", line_width=1.5)

# Filled polygon using the fill colour
closed_nodes = sin_nodes + [(float(x[-1]), 0.0), (float(x[0]), 0.0)]
tf5.draw(closed_nodes, fill="myfill", draw="none", cycle=True)

print(tf5.generate_tikz())

### 2.6 Filled paths and patterns

Pass `fill=` and/or `pattern=` to `draw()` to create shaded regions.

In [ ]:
tf6 = tz.TikzFigure()

# Shaded area under sin curve (closed path)
area_nodes = sin_nodes + [(float(x[-1]), 0.0), (float(x[0]), 0.0)]
tf6.draw(area_nodes, fill="steelblue!20", draw="none", cycle=True)

# Hatched region using a pattern
cos_area = cos_nodes + [(float(x[-1]), 0.0), (float(x[0]), 0.0)]
tf6.draw(
    cos_area,
    pattern="north east lines",
    pattern_color="tomato",
    draw="none",
    cycle=True,
)

# Curves on top
tf6.draw(sin_nodes, color="steelblue", line_width=1.5)
tf6.draw(cos_nodes, color="tomato", line_width=1.2)

print(tf6.generate_tikz())

### 2.7 Layers in `TikzFigure`

The `layer=` parameter on every drawing call controls render order.
Lower-numbered layers are drawn first (behind), higher layers on top.

In [ ]:
tf7 = tz.TikzFigure()

# layer 0: background fill (drawn first)
tf7.rectangle((0, -1.2), (2 * np.pi, 1.2), fill="gray!8", draw="gray!30", layer=0)

# layer 1: shaded area
area = sin_nodes + [(float(x[-1]), 0), (float(x[0]), 0)]
tf7.draw(area, fill="steelblue!25", draw="none", cycle=True, layer=1)

# layer 2: the curve (drawn last, on top)
tf7.draw(sin_nodes, color="steelblue", line_width=2.0, layer=2)
tf7.add_node(np.pi / 2, 1.15, content=r"$\sin(x)$", color="steelblue", layer=2)

print(tf7.generate_tikz())

### 2.8 Escaping to raw TikZ code

For anything not yet covered by the API, use `add_raw()` to inject verbatim TikZ.

In [ ]:
tf8 = tz.TikzFigure()
tf8.draw(sin_nodes, color="steelblue", line_width=1.5)

# Inject custom TikZ — a dashed grid line
tf8.add_raw(r"\draw[gray!40, dashed] (0, 0) -- (6.28, 0);")

# Annotation with arrow using raw TikZ
tf8.add_raw(
    r"\draw[->, gray] (1.0, 0.6) -- (1.57, 1.0) node[right, font=\small] {peak};"
)

print(tf8.generate_tikz())

### 2.9 Putting it all together — a complete figure

Combine paths, shapes, nodes, and colours into a single publication-ready figure.

In [ ]:
tf_final = tz.TikzFigure(figsize=(12, 7))

# --- colours ---
tf_final.colorlet("cblue", "blue!65!white")
tf_final.colorlet("cred", "red!75!black")

# --- background ---
tf_final.rectangle((0, -1.3), (2 * np.pi, 1.3), fill="gray!5", draw="gray!30")

# --- zero axis ---
tf_final.line((0, 0), (2 * np.pi, 0), color="gray!60", dash_pattern="on 2pt off 2pt")

# --- shaded area between curves ---
# approximate: shade where sin > cos (first half)
x_half = x[x <= np.pi]
upper = np.sin(x_half)
lower = np.cos(x_half)
region = [(float(xi), float(u)) for xi, u in zip(x_half, upper)] + [
    (float(xi), float(l)) for xi, l in zip(reversed(x_half), reversed(lower))
]
tf_final.draw(region, fill="cblue!20", draw="none", cycle=True)

# --- curves ---
tf_final.draw(sin_nodes, color="cblue", line_width=1.8)
tf_final.draw(cos_nodes, color="cred", line_width=1.5)

# --- markers at key points ---
tf_final.circle((np.pi / 2, 1.0), radius=0.08, fill="cblue", draw="none")
tf_final.circle((np.pi, 0.0), radius=0.08, fill="cblue", draw="none")
tf_final.circle((0, 1.0), radius=0.08, fill="cred", draw="none")

# --- labels ---
tf_final.add_node(
    np.pi / 2 + 0.3, 1.05, content=r"$\sin(x)$", color="cblue", anchor="west"
)
tf_final.add_node(0.2, 1.1, content=r"$\cos(x)$", color="cred", anchor="west")

# --- save and show ---
with open("complete_figure.tex", "w") as f:
    f.write(tf_final.generate_tikz())
print("Saved complete_figure.tex")
print()
print(tf_final.generate_tikz())

In [ ]:
# Renders to PDF (requires pdflatex):
tf_final.show()

### 2.10 Embedding in a LaTeX document

The generated code is a standalone `tikzpicture` environment.  
Drop it into any LaTeX document:

```latex
\usepackage{tikz}

\begin{figure}[h]
  \centering
  \input{complete_figure.tex}
  \caption{Trigonometric functions with shaded region.}
  \label{fig:trig}
\end{figure}
```

Or compile a standalone PDF with `tikzfigure`'s `generate_standalone()` method:

```python
standalone_src = tf_final.generate_standalone()
with open('standalone.tex', 'w') as f:
    f.write(standalone_src)
# Then: pdflatex standalone.tex
```

---
## Summary

### Canvas → TikZ workflow
```python
canvas, ax = Canvas.subplots(width='10cm', ratio=0.6)
ax.plot(x, y, color='steelblue', line_width=1.5)
tikz = canvas.plot(backend='tikzfigure')
print(tikz.generate_tikz())       # inspect LaTeX
tikz.show()                       # render (needs pdflatex)
```

### Direct `tikzfigure` API — key methods

| Method | Purpose |
|---|---|
| `tf.draw(nodes, color=, line_width=, fill=, ...)` | Path through coordinate list |
| `tf.line(start, end, arrows='->', ...)` | Straight line segment |
| `tf.rectangle(corner1, corner2, fill=, draw=, ...)` | Rectangle |
| `tf.circle(center, radius, fill=, ...)` | Circle |
| `tf.arc(start, start_angle, end_angle, radius, ...)` | Arc |
| `tf.add_node(x, y, content=, shape=, fill=, ...)` | Labelled node |
| `tf.colorlet(name, color_expr)` | Define named colour |
| `tf.add_raw(tikz_code)` | Inject verbatim TikZ |
| `tf.generate_tikz()` | Return LaTeX string |
| `tf.show()` | Compile + display (needs `pdflatex`) |

### TikZ colour syntax cheatsheet
| Expression | Meaning |
|---|---|
| `'red'`, `'blue'`, `'green'` | Standard colours |
| `'blue!70!white'` | 70% blue + 30% white |
| `'red!80!black'` | 80% red + 20% black |
| `'blue!50!red'` | 50% blend |
| `'gray!20'` | 20% gray (80% white) |